# Spotify Popularity Predictor: REST API Documentation

**Serving the trained popularity model as a web service with FastAPI**

| | |
|---|---|
| **Companion to** | `spotify1_formatted.ipynb` (analysis and model training) |
| **Source file** | `app.py` |
| **Framework** | [FastAPI](https://fastapi.tiangolo.com/) with [Pydantic](https://docs.pydantic.dev/) validation |
| **Model** | Random Forest regressor (scikit-learn), loaded from `popularity_model.pkl` |
| **Output** | Predicted popularity score on a 0–100 scale |

---

## Table of Contents

1. [Purpose and Architecture](#1.-Purpose-and-Architecture)
2. [Prerequisites](#2.-Prerequisites)
3. [Loading the Model Artifacts](#3.-Loading-the-Model-Artifacts)
4. [Request Schema](#4.-Request-Schema)
5. [Prediction Logic](#5.-Prediction-Logic)
6. [API Endpoints](#6.-API-Endpoints)
7. [Testing the API](#7.-Testing-the-API)
8. [Running the Server](#8.-Running-the-Server)
9. [Limitations and Production Considerations](#9.-Limitations-and-Production-Considerations)

## 1. Purpose and Architecture

The analysis notebook trained a model that predicts a track's Spotify popularity from seven audio features plus its genre. This API exposes that model over HTTP so that other programs (a web page, a mobile app, a script) can request predictions without needing Python, scikit-learn, or the training data.

### Request flow

```
Client ──► POST /predict (JSON) ──► Pydantic validates input
                                         │
                                         ▼
                        genre text ──► genre_means.pkl lookup ──► genre_encoded (number)
                                         │
                                         ▼
                 audio features + genre_encoded ──► popularity_model.pkl
                                         │
                                         ▼
Client ◄── JSON: predicted_popularity, genre_used, genre_encoded_value
```

### Why two saved files are needed

| File | Contents | Role at inference time |
|---|---|---|
| `popularity_model.pkl` | The trained Random Forest (100 trees) | Produces the prediction |
| `genre_means.pkl` | Mapping of each of the 113 genres to its mean popularity in the training set | Converts the genre name into the numeric `genre_encoded` value the model was trained on |

The model was trained on a **numeric** genre value (target mean encoding), so it cannot interpret genre text directly. The API must repeat the exact same encoding used in training.

## 2. Prerequisites

Install the required packages in the same environment used to train the model:

```bash
uv pip install fastapi uvicorn pydantic pandas scikit-learn joblib httpx
```

> ⚠️ **Version compatibility:** pickled scikit-learn models can fail to load, or behave differently, if the library version differs from the one used for training. Run the API in the same environment (or with the same pinned `scikit-learn` version) as the training notebook.

`httpx` is only needed here because FastAPI's built-in test client uses it (Section 7).

**Working directory:** this notebook expects `popularity_model.pkl` and `genre_means.pkl` to be in the same folder as the notebook.

In [1]:
!uv pip install fastapi uvicorn pydantic pandas scikit-learn joblib httpx

Resolved 27 packages in 13.14s
Prepared 4 packages in 842ms
Installed 16 packages in 170ms
 + annotated-doc==0.0.6
 + annotated-types==0.8.0
 + anyio==4.16.0
 + certifi==2026.10.23
 + click==8.5.1
 + fastapi==0.162.5
 + h11==0.16.0
 + httpcore==1.0.9
 + httpx==0.28.1
 + idna==3.20
 + opentelemetry-api==1.45.2
 + pydantic==2.14.3
 + pydantic-core==2.50.4
 + starlette==1.10.0
 + typing-inspection==0.4.5
 + uvicorn==0.54.0


In [2]:
import warnings
from pathlib import Path

warnings.filterwarnings("ignore", message=".*httpx.*")  # harmless TestClient notice

import joblib
import pandas as pd
from fastapi import FastAPI
from fastapi.testclient import TestClient
from pydantic import BaseModel

# In app.py this is Path(__file__).resolve().parent. Notebooks have no __file__,
# so the current working directory is used instead.
BASE_DIR = Path.cwd()

## 3. Loading the Model Artifacts

The model and genre mapping are loaded **once, when the application starts**, and then reused for every request. Loading inside the request handler would reload a large file on each call and make every response slow.

In [3]:
model = joblib.load(BASE_DIR / "popularity_model.pkl")
genre_means = joblib.load(BASE_DIR / "genre_means.pkl")

print(f"Model type          : {type(model).__name__}")
print(f"Number of trees     : {model.n_estimators}")
print(f"Genres in mapping   : {len(genre_means)}")
print(f"Fallback (mean)     : {genre_means.mean():.2f}")

Model type          : RandomForestRegressor
Number of trees     : 100
Genres in mapping   : 113
Fallback (mean)     : 33.92


### Consistency check: feature order

A model trained on a DataFrame remembers the **names and order** of its input columns. The API must build its input in exactly that order. This cell makes the expectation explicit and fails loudly if the model and the API ever disagree.

In [4]:
FEATURE_ORDER = [
    "danceability", "energy", "loudness", "valence",
    "tempo", "acousticness", "speechiness", "genre_encoded",
]

assert list(model.feature_names_in_) == FEATURE_ORDER, "Model features do not match the API"
print("Feature order matches the trained model:")
print(list(model.feature_names_in_))

Feature order matches the trained model:
['danceability', 'energy', 'loudness', 'valence', 'tempo', 'acousticness', 'speechiness', 'genre_encoded']


## 4. Request Schema

`SongFeatures` is a Pydantic model that defines what a valid request looks like. FastAPI uses it to:

- **Validate** incoming JSON, rejecting a request with a missing field or a wrong type (e.g. text where a number is expected) with a clear `422` error before any prediction code runs.
- **Generate interactive documentation** automatically at `/docs`.

| Field | Type | Meaning | Typical range |
|---|---|---|---|
| `danceability` | float | Suitability for dancing | 0.0 – 1.0 |
| `energy` | float | Intensity and activity | 0.0 – 1.0 |
| `loudness` | float | Overall loudness in dB | about −60 – 0 |
| `valence` | float | Musical positiveness | 0.0 – 1.0 |
| `tempo` | float | Beats per minute | about 50 – 220 |
| `acousticness` | float | Confidence the track is acoustic | 0.0 – 1.0 |
| `speechiness` | float | Presence of spoken words | 0.0 – 1.0 |
| `genre` | str | Genre name, e.g. `"k-pop"` | one of the 113 training genres |

> The ranges above are descriptive only. The schema checks **types**, not ranges (see Section 9).

In [5]:
class SongFeatures(BaseModel):
    danceability: float
    energy: float
    loudness: float
    valence: float
    tempo: float
    acousticness: float
    speechiness: float
    genre: str

## 5. Prediction Logic

Prediction happens in three steps:

1. **Encode the genre.** Look up the genre in `genre_means`. If the genre was not seen in training, fall back to the **overall mean** of the mapping so the request still succeeds instead of failing.
2. **Build the feature row.** Create a one-row DataFrame with columns in `FEATURE_ORDER`.
3. **Predict.** Run the model and round the result to two decimals.

The logic is kept in a plain function, separate from the web layer, so it can be tested and reused without starting a server.

In [6]:
def predict_from_features(song: SongFeatures) -> dict:
    # Step 1: encode the genre; unseen genres fall back to the overall average
    genre_encoded = genre_means.get(song.genre, genre_means.mean())

    # Step 2: build the feature row in the exact order the model expects
    input_data = pd.DataFrame([{
        "danceability": song.danceability,
        "energy": song.energy,
        "loudness": song.loudness,
        "valence": song.valence,
        "tempo": song.tempo,
        "acousticness": song.acousticness,
        "speechiness": song.speechiness,
        "genre_encoded": genre_encoded,
    }])[FEATURE_ORDER]

    # Step 3: predict
    prediction = model.predict(input_data)[0]

    return {
        "predicted_popularity": round(float(prediction), 2),
        "genre_used": song.genre,
        "genre_encoded_value": round(float(genre_encoded), 2),
    }

## 6. API Endpoints

| Method | Path | Description |
|---|---|---|
| `GET` | `/` | Health check. Confirms the service is running |
| `POST` | `/predict` | Accepts a `SongFeatures` JSON body and returns the predicted popularity |
| `GET` | `/docs` | Auto-generated interactive documentation (Swagger UI) |

### `POST /predict`: example

**Request body**
```json
{
  "danceability": 0.75, "energy": 0.80, "loudness": -5.0,
  "valence": 0.60, "tempo": 120.0, "acousticness": 0.10,
  "speechiness": 0.05, "genre": "k-pop"
}
```

**Response**
```json
{
  "predicted_popularity": 68.84,
  "genre_used": "k-pop",
  "genre_encoded_value": 59.28
}
```

The response includes `genre_used` and `genre_encoded_value` so the caller can see exactly which genre value the model received. This is useful for spotting an unseen genre that triggered the fallback.

In [7]:
app = FastAPI(title="Spotify Popularity Predictor")


@app.get("/")
def home():
    # Health-check endpoint
    return {"message": "Spotify Popularity Predictor API is running"}


@app.post("/predict")
def predict_popularity(song: SongFeatures):
    # Predict popularity for one track
    return predict_from_features(song)

## 7. Testing the API

Instead of starting a real server, FastAPI's `TestClient` sends requests directly to the application inside this notebook. This exercises the same routing, validation, and prediction code that runs in production.

### 7.1 Health check

In [8]:
client = TestClient(app)

response = client.get("/")
print("Status:", response.status_code)
print("Body  :", response.json())

Status: 200
Body  : {'message': 'Spotify Popularity Predictor API is running'}


### 7.2 Valid prediction
A typical request for a known genre.

In [9]:
payload = {
    "danceability": 0.75, "energy": 0.80, "loudness": -5.0,
    "valence": 0.60, "tempo": 120.0, "acousticness": 0.10,
    "speechiness": 0.05, "genre": "k-pop",
}

response = client.post("/predict", json=payload)
print("Status:", response.status_code)
print("Body  :", response.json())

Status: 200
Body  : {'predicted_popularity': 68.84, 'genre_used': 'k-pop', 'genre_encoded_value': 59.28}


### 7.3 Genre sensitivity
The analysis showed that genre dominates popularity. Sending the **same audio features** with different genres should produce clearly different predictions, which confirms the API applies the genre mapping correctly.

In [10]:
rows = []
for genre in ["k-pop", "pop-film", "metal", "jazz", "iranian"]:
    result = client.post("/predict", json={**payload, "genre": genre}).json()
    rows.append({
        "genre": genre,
        "genre_encoded_value": result["genre_encoded_value"],
        "predicted_popularity": result["predicted_popularity"],
    })

pd.DataFrame(rows).set_index("genre")

,genre_encoded_value,predicted_popularity
genre,,
k-pop,59.28,68.84
pop-film,59.12,60.29
metal,56.00,51.86
jazz,9.50,11.14
iranian,2.26,5.38


### 7.4 Unseen genre (fallback behaviour)
A genre that does not exist in the training data should not cause an error. The API uses the overall mean instead, and `genre_encoded_value` reveals that the fallback was applied.

In [11]:
response = client.post("/predict", json={**payload, "genre": "not-a-real-genre"})
print("Status:", response.status_code)
print("Body  :", response.json())
print(f"\nOverall genre mean (fallback): {genre_means.mean():.2f}")

Status: 200
Body  : {'predicted_popularity': 43.96, 'genre_used': 'not-a-real-genre', 'genre_encoded_value': 33.92}

Overall genre mean (fallback): 33.92


### 7.5 Invalid input
A request with a missing field or the wrong type is rejected by the schema with HTTP **422** before reaching the model.

In [12]:
# Missing the required "genre" field and a non-numeric "tempo"
bad_payload = {k: v for k, v in payload.items() if k != "genre"}
bad_payload["tempo"] = "fast"

response = client.post("/predict", json=bad_payload)
print("Status:", response.status_code)
for err in response.json()["detail"]:
    print(f"- field {err['loc'][-1]!r}: {err['msg']}")

Status: 422
- field 'tempo': Input should be a valid number, unable to parse string as a number
- field 'genre': Field required


## 8. Running the Server

The cells above test the API inside the notebook. To serve it to real clients, run `app.py` with **Uvicorn** from a terminal (not from the notebook):

```bat
cd C:\Experiemnts\Experiment_1_Spotify
uvicorn app:app --reload
```

- `app:app` means "the object named `app` inside the file `app.py`".
- `--reload` restarts the server when the code changes (use it in development only).
- The server starts on `http://127.0.0.1:8000`.

### Calling the running API

**Interactive docs (easiest):** open `http://127.0.0.1:8000/docs`, expand `POST /predict`, click **Try it out**, and execute.

**Python:**
```python
import requests

r = requests.post("http://127.0.0.1:8000/predict", json={
    "danceability": 0.75, "energy": 0.80, "loudness": -5.0,
    "valence": 0.60, "tempo": 120.0, "acousticness": 0.10,
    "speechiness": 0.05, "genre": "k-pop",
})
print(r.json())
```

**curl (Windows Command Prompt):**
```bat
curl -X POST http://127.0.0.1:8000/predict -H "Content-Type: application/json" -d "{\"danceability\":0.75,\"energy\":0.80,\"loudness\":-5.0,\"valence\":0.60,\"tempo\":120.0,\"acousticness\":0.10,\"speechiness\":0.05,\"genre\":\"k-pop\"}"
```

## 9. Limitations and Production Considerations

### Model limitations (inherited from the analysis)
- The model explains about **46% of the variance** (R² ≈ 0.46, average error about 10 points). Treat predictions as rough estimates, not exact values.
- Popularity also depends on artist fame, marketing, and release timing, none of which are inputs.
- Genre is the dominant input. A song tagged with a low-popularity genre receives a low prediction regardless of its audio features.

### API improvements to consider
| Area | Current behaviour | Suggested improvement |
|---|---|---|
| **Input validation** | Checks types only | Add range checks with `Field(ge=0, le=1)` for the 0–1 features, and sensible bounds for `loudness` and `tempo` |
| **Unseen genres** | Silently uses the overall mean | Return a warning field or a `400` error, or offer a `/genres` endpoint listing valid values |
| **Genre matching** | Case- and spelling-sensitive (`"K-Pop"` is treated as unseen) | Normalize with `.strip().lower()` before lookup |
| **Model loading** | Relative file paths | Resolve paths from the file location (`Path(__file__).resolve().parent`) |
| **Deployment** | Single process, development server | Containerize (Docker), pin dependency versions, and run behind a production server |
| **Monitoring** | None | Add request logging and track how often the genre fallback is used |
| **Security** | No authentication | Add an API key or authentication before exposing it publicly |

### Relationship to `app.py`
This notebook **documents and tests** the service. For deployment, keep using `app.py`, which contains the same loading, schema, and endpoint logic as a standalone script.